#### Sentinel-2 滇池区域数据下载
准备工作：   
(1) 注册gee账户     
(1) google云平台注册云端项目     
(2) google网盘      

In [1]:
import ee  
import geemap   
ee.Authenticate(auth_mode='localhost')   
ee.Initialize(project='my-earth-engine-2026')    


c:\Users\HP\miniconda3\envs\gis-course\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python
c:\Users\HP\miniconda3\envs\gis-course\lib\site-packages\ee\deprecation.py:140: UserWarning: Unable to initialize deprecated assets: [ASN1: NOT_ENOUGH_DATA] not enough data (_ssl.c:4040)
  warnings.warn(f'Unable to initialize deprecated assets: {e}')


#### 参数设置  
滇池空间范围：[102.55, 24.6, 102.82, 25.03]

In [2]:
# ── 研究区域：滇池 ────
region = ee.Geometry.Rectangle([102.55, 24.6, 102.82, 25.03])

# ── 数据集与波段 ─────
dset  = 'COPERNICUS/S2_SR_HARMONIZED'
band_sel   = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']  # Blue,Green,Red,NIR,SWIR1,SWIR2
band_vis   = ['B4', 'B3', 'B2']                      # 真彩色

print(f'研究区面积: {region.area().getInfo() / 1e6:.1f} km²')  # type: ignore



研究区面积: 1303.0 km²


In [3]:
## sentinel-2
img_col = (ee.ImageCollection(dset)
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
           .filterBounds(region)
           .filterDate('2025-07-01', '2025-12-31')
           .sort('CLOUDY_PIXEL_PERCENTAGE')           
           )
img_col


In [4]:
img_col = img_col.filter(ee.Filter.contains('.geo', region))  
img_sel = img_col.first().clip(region).select(band_sel) 
img_sel


In [5]:
Map = geemap.Map()
Map.centerObject(region, 6)
Map.addLayer(img_sel, {'bands': band_vis, 'min': 0, 'max': 2000}, \
             'Sentinel-2', True, 1)
# Map


In [ ]:
# # type: ignore
# task = ee.batch.Export.image.toDrive(   
#     image=img_sel,
#     description='s2_20251206',
#     folder='gee-down',
#     scale=20,     
#     fileFormat='GeoTIFF',
#     region=region)
# task.start()
# print('Export task started:', task.status())  


Export task started: {'state': 'READY', 'description': 's2_20251206', 'priority': 100, 'creation_timestamp_ms': 1790751558564, 'update_timestamp_ms': 1790751558564, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_IMAGE', 'id': 'VGBGFLN3D73VNJ37EJIOFURH', 'name': 'projects/my-earth-engine-2026/operations/VGBGFLN3D73VNJ37EJIOFURH'}
